#### QUERY `GIZMO.BRONZE.CUSTOMERS_DELTA`
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: BRONZE
3. VIEW NAME: CUSTOMERS_DELTA

In [0]:
%python
customers_df = spark.table("GIZMO.BRONZE.CUSTOMERS_DELTA").distinct().orderBy('customer_id').filter('customer_id IS NOT NULL')
display(customers_df)
display(f'Record Count: {customers_df.count()}')

#### DEFINE UDF TO EXTRACT FIRST NAME FROM CUSTOMER_NAME

In [0]:
def extract_first_name(customer_name):
    return customer_name.split(' ')[0] if customer_name else None

#### DEFINE UDF TO EXTRACT LAST NAME FROM CUSTOMER_NAME

In [0]:
def extract_last_name(customer_name):
    if customer_name and ' ' in customer_name:
        return customer_name.split(' ', 1)[1]
    return None

#### REGISTER `EXTRACT_FIRST_NAME`, `EXTRACT_LAST_NAME` AS UDF

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Register the function as a UDF
first_name_udf = udf(extract_first_name, StringType())
last_name_udf = udf(extract_last_name, StringType())

In [0]:
customers_name_extract_df = (customers_df.select('customer_id', 'customer_name','date_of_birth', 'email', 'member_since', 
                                            'telephone', 'created_timestamp', 'file_name', 'file_path')
                                      .withColumn('first_name', first_name_udf('customer_name'))
                                      .withColumn('last_name', last_name_udf('customer_name'))
                                      .drop('customer_name')
                                      )
display(customers_name_extract_df)

#### SELECT REQUIRED COLUMNS
1. **`customer_id`, `first_name`, `last_name`, `date_of_birth`, `email`, `member_since`, `telephone`, `created_timestamp`, `file_name`, `file_path`**
2. SORT THE DATAFRAME BASED ON `CUSTOMER_ID` IN ASCENDING
3. FILTER OUT `CUSTOMER_ID` NOT NULL


In [0]:
customers_selected_cols_df = customers_name_extract_df.select('customer_id', 'first_name', 'last_name', 'date_of_birth', 'email', 'member_since', 
                                            'telephone', 'created_timestamp', 'file_name', 'file_path')
display(customers_selected_cols_df)

#### NEED TO LATEST CUSTOMERS BASED ON CREATED_TIMESTAMP

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

windowSpec = Window.partitionBy("customer_id").orderBy(col("created_timestamp").desc())

ranked_customer_df = customers_selected_cols_df.withColumn("rank", rank().over(windowSpec)).filter('rank = 1')
display(ranked_customer_df)

#### WRITE CUSTOMERS TO DELTA FORMAT

In [0]:
ranked_customer_df.writeTo('gizmo.silver.customers_delta').createOrReplace()

#### QUERY AND VALIDATE `GIZMO.SILVER.CUSTOMERS_DELTA`

In [0]:
%python
customers_count_df = spark.sql('''SELECT * FROM GIZMO.SILVER.CUSTOMERS_DELTA''');
print(f'Row Count: {customers_count_df.count()}')

## 📝 CAPTURE AUDIT & OBSERVABILITY MECHANISM

Track and log every data pipeline run for transparency, traceability, and operational monitoring.  
This section ensures all data loads are auditable and pipeline health is observable.

In [0]:
%python
from pyspark.sql import Row
from datetime import datetime
import uuid
import sys
import traceback
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType, LongType, TimestampType

# =====================================================================
# 1. INITIALIZE PIPELINE METADATA
# =====================================================================
load_start_time = datetime.now()
pipeline_name = 'PySpark-01.TransformCustomers'

status = "SUCCESS"
message = "Loaded Customers data into silver target table"  # Fixed descriptive message
record_count = 0

try:
    # =====================================================================
    # 2. CORE ETL LOGIC
    # =====================================================================
    
    # Step A: Extract Data 
    # REMOVED: .orderBy('customer_id') - Sorting here degrades cluster performance via unnecessary shuffles
    customers_df = spark.table("GIZMO.BRONZE.CUSTOMERS_DELTA") \
                        .filter(F.col('customer_id').isNotNull()) \
                        .distinct()
    
    # Step B: Target Transformation / Action Execution
    # Executing actual target write operations here justifies the action cost below
    # customers_df.write.mode("overwrite").saveAsTable("GIZMO.SILVER.CUSTOMERS")
    
    # Step C: Capture final evaluated target record count
    record_count = customers_df.count()
    
    # =====================================================================

except Exception as e:
    # 3. EXCEPTION HANDLING
    status = "FAILED"
    
    exc_type, exc_value, exc_tb = sys.exc_info()
    error_details = traceback.format_exception_only(exc_type, exc_value)[0].strip()
    message = f"Pipeline failed! Error: {error_details}"
    record_count = -1 

finally:
    # =====================================================================
    # 4. AUDIT & LOGGING
    # =====================================================================
    load_end_time = datetime.now()
    
    # Use START time for historical continuity so runs wrapping past midnight stay grouped
    current_date = load_start_time.date() 

    # Step A: Calculate Sequential Run ID for Today
    try:
        # Cast column to String to ensure strict type matching against Python's str(current_date)
        max_run_df = spark.table("GIZMO.AUDIT.AUDIT_LOGS") \
            .filter(
                (F.col("event_time").cast("string") == F.lit(str(current_date))) & 
                (F.col("pipeline_name") == F.lit(pipeline_name))
            ) \
            .select(F.max(F.col("run_id").cast("int")).alias("max_id"))
        
        max_id_row = max_run_df.collect()[0]
        next_run_int = (max_id_row["max_id"] + 1) if max_id_row["max_id"] is not None else 1
    except Exception as audit_ex:
        # If audit table doesn't exist or fails, default to 1 safely
        next_run_int = 1

    run_id_str = f"{next_run_int:02d}"

    # Step B: Secure Notebook Cluster Context Metadata safely
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        notebook_path = context.notebookPath().get()
        user_name = context.tags().apply("user")
    except Exception:
        notebook_path = "Unknown/Local"
        user_name = "System"

    # Step C: Package the metadata tracking Row
    log_entry = Row(
        log_id=str(uuid.uuid4()),
        run_id=run_id_str,                
        event_time=current_date,          
        event_type="FULL LOAD",
        source_table='GIZMO.BRONZE.CUSTOMERS_DELTA',
        target_table="GIZMO.SILVER.CUSTOMERS_DELTA",
        record_count=record_count,
        status=status,                    
        message=message,                   
        user_name=user_name,
        notebook_path=notebook_path,
        pipeline_name=pipeline_name,
        load_start_time=load_start_time,  
        load_end_time=load_end_time       
    )

    # Step D: Explicit Schema DDL Matching
    log_schema = StructType([
        StructField("log_id", StringType(), True),
        StructField("run_id", StringType(), True),  
        StructField("event_time", DateType(), True),
        StructField("event_type", StringType(), True),
        StructField("source_table", StringType(), True),
        StructField("target_table", StringType(), True),
        StructField("record_count", LongType(), True),
        StructField("status", StringType(), True),
        StructField("message", StringType(), True),
        StructField("user_name", StringType(), True),
        StructField("notebook_path", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("load_start_time", TimestampType(), True),
        StructField("load_end_time", TimestampType(), True)
    ])

    # Step E: Write transactional logging entry to Delta table
    try:
        log_entry_df = spark.createDataFrame([log_entry], schema=log_schema)
        log_entry_df.write.format("delta").mode("append").saveAsTable("GIZMO.AUDIT.AUDIT_LOGS")
        print(f"[AUDIT LOGGED] Status: {status} | Run ID: {run_id_str} | Count: {record_count}")
    except Exception as log_write_err:
        print(f"[CRITICAL] Audit logging failed to write to disk: {str(log_write_err)}")

    # Step F: Force a hard stop exception for workflow orchestrators if pipeline failed
    if status == "FAILED":
        raise RuntimeError(message)

In [0]:
%python
dbutils.notebook.exit("CUSTOMERS LOADED INTO GIZMO.SILVER.CUSTOMERS")

#### VALIDATE AUDIT TABLE RECORD COUNT `GIZMO.BRONZE.AUDIT_LOGS`

In [0]:
%sql
SELECT 
  run_id, 
  event_time, 
  pipeline_name, 
  record_count,
  date_format(FROM_UTC_TIMESTAMP(load_start_time, 'Asia/Kolkata'), 'yyyy-MM-dd HH:mm:ss') AS load_start_time_ist, 
  date_format(FROM_UTC_TIMESTAMP(load_end_time, 'Asia/Kolkata'), 'yyyy-MM-dd HH:mm:ss') AS load_end_time_ist
FROM GIZMO.AUDIT.AUDIT_LOGS
WHERE pipeline_name = 'PySpark-01.TransformCustomers'
ORDER BY pipeline_name, event_time DESC, run_id ASC;